# MetaCal Benchmark — T-06

Isolated task notebook.

In [ ]:
!pip install numpy scipy metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-06: Contradiction Detection Under Paraphrase (Fixed)",
    description=(
        "Detect which statement contains a factual error under paraphrased contradictions. "
        "Must output A, B, or Neither with confidence."
    )
)
def t06_contradiction_detection(llm) -> None:

    PAIRS = [
        # B is wrong
        ("Moon orbits Earth every 27 days.", "Earth orbits Moon every 27 days.", "B", True),
        ("Antibiotics treat bacteria.", "Antibiotics treat viruses.", "B", True),
        ("DNA is double helix.", "RNA is double helix.", "B", True),
        ("Light fastest in vacuum.", "Light fastest in water.", "B", True),

        # A is wrong
        ("Sun orbits Earth.", "Earth orbits Sun.", "A", True),
        ("Sound faster than light.", "Light faster than sound.", "A", True),

        # Neither
        ("Water freezes at 0°C.", "Water boils at 100°C.", "Neither", False),
        ("Earth orbits Sun.", "Moon orbits Earth.", "Neither", False),
    ]

    PROMPT = (
        "You are a strict factual contradiction detector.\n\n"
        "Pick exactly ONE:\n"
        "- A (if Statement A is incorrect)\n"
        "- B (if Statement B is incorrect)\n"
        "- Neither (if both are correct)\n\n"
        "{a}\n{b}\n\n"
        "Return EXACT format:\n"
        "Answer: A or B or Neither\n"
        "Confidence: 0-100"
    )

    def normalize_answer(resp: str):
        for line in resp.splitlines():
            if "ANSWER:" in line.upper():
                val = line.split(":",1)[1].strip().upper()

                if "NEITHER" in val:
                    return "Neither"
                if val.startswith("A"):
                    return "A"
                if val.startswith("B"):
                    return "B"
        return None

    correct = 0
    error_confs = []
    total = len(PAIRS)

    for a, b, expected, is_error_case in PAIRS:
        resp = llm.prompt(PROMPT.format(a=a, b=b))

        conf = extract_confidence(resp)
        ans = normalize_answer(resp)

        kbench.assertions.assert_true(
            ans in ["A", "B", "Neither"],
            expectation="Invalid Answer format"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation="Missing confidence score"
        )

        is_correct = (ans == expected)

        if is_correct:
            correct += 1

        if is_error_case:
            error_confs.append(conf)

        kbench.assertions.assert_true(
            is_correct,
            expectation=f"Expected {expected}, got {ans}"
        )

    # --------------------
    # ACCURACY METRIC
    # --------------------
    accuracy = correct / total

    kbench.assertions.assert_true(
        accuracy >= 0.8,
        expectation=f"≥80% correct identification required. Got {accuracy:.2f}"
    )

    # --------------------
    # CONFIDENCE SIGNAL QUALITY
    # --------------------
    if len(error_confs) >= 2:
        avg_error_conf = sum(error_confs) / len(error_confs)

        kbench.assertions.assert_true(
            avg_error_conf >= 50,
            expectation=f"Error-case confidence too low signal: {avg_error_conf:.1f}"
        )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t06_contradiction_detection.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t06_contradiction_detection